# Test Model Loader on Colab

This notebook tests the generic MoE model loader with Qwen3-VL-30B on Google Colab.

**Requirements:**
- Google Colab with A100 GPU (40GB VRAM)
- Hugging Face token (if model requires authentication)


## 1. Check GPU and Memory

In [ ]:
import torch
import subprocess

print("=" * 60)
print("GPU Information")
print("=" * 60)

if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ VRAM: {vram_gb:.1f}GB")
    print(f"✅ PyTorch Version: {torch.__version__}")
    print(f"✅ CUDA Available: {torch.version.cuda}")
else:
    print("❌ No GPU detected!")

# Check free memory
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'], 
                       capture_output=True, text=True)
if result.returncode == 0:
    free_vram = int(result.stdout.strip()) / 1024
    print(f"✅ Free VRAM: {free_vram:.1f}GB")
else:
    print("⚠️  Could not check free VRAM")

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers torch datasets accelerate
print("✅ Dependencies installed")

## 3. Load Model Loader Code

In [ ]:
# Model loader code (from phase_1/scripts/model_loader.py)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Tuple, Optional, Dict, Any
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class ModelConfig:
    """Configuration for different MoE models."""

    MODELS = {
        "qwen3-moe": {
            "model_id": "mistralai/Mixtral-8x7B-Instruct-v0.1",
            "router_gate_path": "block_sparse_moe.gate",
            "num_experts": 8,
            "top_k": 2,
            "is_vision": False,
        },
        "qwen3-vl-30b": {
            "model_id": "Qwen/Qwen3-VL-30B-A3B-Instruct",
            "router_gate_path": "mlp.gate",
            "num_experts": 128,
            "top_k": 6,
            "is_vision": True,
            "note": "Vision-language model - requires custom loading",
        },
        "mixtral-8x7b": {
            "model_id": "mistralai/Mixtral-8x7B-Instruct-v0.1",
            "router_gate_path": "block_sparse_moe.gate",
            "num_experts": 8,
            "top_k": 2,
            "is_vision": False,
        },
        "deepseek-v2-lite": {
            "model_id": "deepseek-ai/DeepSeek-V2-Lite",
            "router_gate_path": "mlp.gate",
            "num_experts": 64,
            "top_k": 6,
            "is_vision": False,
        },
    }

    @classmethod
    def get(cls, model_name: str) -> Dict[str, Any]:
        """Get configuration for a model."""
        name = model_name.lower().replace(" ", "-")
        if name not in cls.MODELS:
            raise ValueError(f"Unknown model: {model_name}. Available: {list(cls.MODELS.keys())}")
        return cls.MODELS[name]


class ModelLoader:
    """Load MoE models with hardware detection and memory optimization."""

    def __init__(self, model_name: str, device: Optional[str] = None, dtype: torch.dtype = torch.float16):
        self.config = ModelConfig.get(model_name)
        self.model_id = self.config["model_id"]
        self.dtype = dtype
        self.device = device or self._detect_device()
        self.model = None
        self.tokenizer = None

        logger.info(f"Initialized {model_name} loader")
        logger.info(f"  Device: {self.device}")
        logger.info(f"  Dtype: {dtype}")
        logger.info(f"  Experts: {self.config['num_experts']}, Top-K: {self.config['top_k']}")

    def _detect_device(self) -> str:
        """Auto-detect available hardware."""
        if torch.cuda.is_available():
            device_name = torch.cuda.get_device_name(0)
            vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            logger.info(f"GPU detected: {device_name} ({vram_gb:.1f}GB VRAM)")
            return "cuda"
        logger.warning("No GPU detected, using CPU (slow)")
        return "cpu"

    def load(self, trust_remote_code: bool = True) -> Tuple[Any, Any, Dict[str, Any]]:
        """Load model and tokenizer."""
        logger.info(f"Loading {self.model_id}...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_id,
            trust_remote_code=trust_remote_code,
            padding_side="left",
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            torch_dtype=self.dtype,
            device_map=self.device,
            trust_remote_code=trust_remote_code,
        )

        self.model.eval()
        logger.info(f"Model loaded successfully")
        logger.info(f"  Total parameters: {self._count_parameters(self.model) / 1e9:.1f}B")

        return self.model, self.tokenizer, self.config

    def get_router_gate(self, layer: Any) -> Optional[Any]:
        """Extract router gate from a layer."""
        gate_path = self.config["router_gate_path"]
        parts = gate_path.split(".")

        current = layer
        for part in parts:
            if hasattr(current, part):
                current = getattr(current, part)
            else:
                return None

        return current if current is not None else None

    @staticmethod
    def _count_parameters(model: Any) -> int:
        """Count total parameters in model."""
        return sum(p.numel() for p in model.parameters())

    def get_model_info(self) -> Dict[str, Any]:
        """Get detailed model information."""
        return {
            "model_id": self.model_id,
            "device": self.device,
            "dtype": str(self.dtype),
            "num_experts": self.config["num_experts"],
            "top_k": self.config["top_k"],
            "is_vision": self.config["is_vision"],
            "total_params": self._count_parameters(self.model) if self.model else None,
        }


print("✅ Model loader code loaded")

## 4. Load MoE Model

In [ ]:
print("Loading MoE model with 4-bit quantization...")
print("This may take a few minutes...\n")

try:
    loader = ModelLoader("mixtral-8x7b", dtype=torch.float16, load_in_4bit=True)
    model, tokenizer, config = loader.load()
    print("\n✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    raise

## 5. Test Model Tokenizer

In [ ]:
test_texts = [
    "Hello, how are you?",
    "Write a Python function to calculate factorial",
    "What is 2 + 2?",
]

print("Testing tokenizer...\n")
for text in test_texts:
    tokens = tokenizer.encode(text)
    print(f"Text: {text}")
    print(f"Tokens: {len(tokens)}")
    print()

print("✅ Tokenizer test passed")

## 6. Test Router Gate Access

In [ ]:
print("Testing router gate access...\n")

if hasattr(model, "model") and hasattr(model.model, "layers"):
    num_layers = len(model.model.layers)
    print(f"Model has {num_layers} layers\n")
    
    gates_found = 0
    for i in range(min(3, num_layers)):  # Test first 3 layers
        layer = model.model.layers[i]
        gate = loader.get_router_gate(layer)
        if gate is not None:
            gates_found += 1
            print(f"✅ Layer {i}: Router gate found")
        else:
            print(f"⚠️  Layer {i}: Router gate not found")
    
    if gates_found > 0:
        print(f"\n✅ Router gates accessible! Found {gates_found}/{min(3, num_layers)}")
    else:
        print("\n⚠️  Could not find router gates (may need architecture-specific handling)")
else:
    print("⚠️  Could not access model layers")

## 7. Quick Forward Pass Test

In [ ]:
print("Testing forward pass...\n")

test_input = "What is 5 + 3?"
tokens = tokenizer.encode(test_input, return_tensors="pt").to("cuda")

print(f"Input: {test_input}")
print(f"Input tokens shape: {tokens.shape}")

with torch.no_grad():
    outputs = model(tokens)

print(f"Output logits shape: {outputs.logits.shape}")
print("\n✅ Forward pass successful!")

## 8. Summary

In [ ]:
print("\n" + "=" * 60)
print("MODEL LOADER TEST SUMMARY")
print("=" * 60)

info = loader.get_model_info()
for key, value in info.items():
    print(f"{key:20s}: {value}")

print("\n" + "=" * 60)
print("✅ All tests passed! Ready for Phase 1 telemetry hooks.")
print("=" * 60)